# NAIP Imagery Tile Service URL Generator

This notebook demonstrates how to process NAIP (National Agriculture Imagery Program) imagery utilizing Google Earth Engine and retrieve the URLs for the tile services.

## Efficiency Note
The Geometry provided below is used to create a Bounding Box (BBOX). This limits the number of tiles produced by strictly defining the Area of Interest (AOI), which significantly improves efficiency (speed and resource usage) compared to processing wider extents.

In [3]:
!pip install earthengine-api ee geemap --quiet

In [5]:
%pip install --upgrade earthengine-api geemap --quiet

import ee
import json

# Initialize the Earth Engine module.
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

Note: you may need to restart the kernel to use updated packages.


In [6]:
# 1. Define the GeoJSON Area of Interest
roi_json = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {
        "name": "mortalitree_czu_download_bbox"
      },
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [
              -122.346241721068,
              37.0180037353093
            ],
            [
              -122.087073088937,
              37.0180037353093
            ],
            [
              -122.087073088937,
              37.2720781887031
            ],
            [
              -122.346241721068,
              37.2720781887031
            ],
            [
              -122.346241721068,
              37.0180037353093
            ]
          ]
        ]
      }
    }
  ]
}

# Convert the GeoJSON object to an EE Geometry
geometry = ee.FeatureCollection(roi_json).geometry()

In [7]:
# 2. Filter collection by geometry and broad date range
dataset = ee.ImageCollection('USDA/NAIP/DOQQ') \
    .filterBounds(geometry) \
    .filterDate('2000-01-01', '2025-12-31')

# 3. Get distinct years
# We execute this client-side (getInfo) to iterate in Python
years = dataset.aggregate_array('system:time_start') \
    .map(lambda time: ee.Date(time).get('year')) \
    .distinct() \
    .sort()

year_list = years.getInfo()
print(f"Found NAIP imagery for years: {year_list}")

Found NAIP imagery for years: [2003, 2004, 2005, 2006, 2009, 2010, 2012, 2014, 2016, 2018, 2020, 2022]


In [8]:
import os
import csv

# Define output file path
output_file = '../output/tile_urls.csv'

# Ensure output directory exists (it should, but good practice)
os.makedirs(os.path.dirname(output_file), exist_ok=True)

print(f"Processing years and saving to {output_file}...")

with open(output_file, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    # Write header
    writer.writerow(["name", "tile_url"])
    
    # Also print header to stdout for visibility
    print('"name","tile_url"')

    for year in year_list:
        yearly_col = dataset.filterDate(
            ee.Date.fromYMD(year, 1, 1), 
            ee.Date.fromYMD(year, 12, 31)
        )
        
        # Create mosaic and clip to geometry
        mosaic = yearly_col.mosaic().clip(geometry)
        
        # Check bands available for this specific year
        try:
            band_names = mosaic.bandNames().getInfo()
        except Exception as e:
            continue

        # Calculate Min/Max statistics for the geometry (100% Stretch)
        stats = mosaic.reduceRegion(
            reducer=ee.Reducer.minMax(),
            geometry=geometry,
            scale=10, # Increased scale slightly to speed up stats on large bbox
            bestEffort=True, 
            maxPixels=1e9
        )
        
        val = stats.getInfo()
        
        # --- A. TRUE COLOR LAYER (RGB) ---
        if set(['R', 'G', 'B']).issubset(band_names):
            viz_rgb = {
              'bands': ['R', 'G', 'B'],
              'min': [val['R_min'], val['G_min'], val['B_min']], 
              'max': [val['R_max'], val['G_max'], val['B_max']]
            }
            
            map_id = mosaic.getMapId(viz_rgb)
            url_format = map_id["tile_fetcher"].url_format
            name = f"gee_naip_{year}"
            
            # Write to file
            writer.writerow([name, url_format])
            # Print to stdout
            print(f'"{name}","{url_format}"')

        # --- B. FALSE COLOR INFRARED LAYER (NRG) ---
        if 'N' in band_names and set(['R', 'G']).issubset(band_names):
            viz_ir = {
                'bands': ['N', 'R', 'G'],
                'min': [val['N_min'], val['R_min'], val['G_min']], 
                'max': [val['N_max'], val['R_max'], val['G_max']]
            }
            
            map_id_ir = mosaic.getMapId(viz_ir)
            url_format_ir = map_id_ir["tile_fetcher"].url_format
            name_ir = f"gee_naip_ir_{year}"
            
            # Write to file
            writer.writerow([name_ir, url_format_ir])
            # Print to stdout
            print(f'"{name_ir}","{url_format_ir}"')

print(f"Successfully saved URLs to {output_file}")

Processing years and saving to ../output/tile_urls.csv...
"name","tile_url"
"gee_naip_2003","https://earthengine.googleapis.com/v1/projects/676236615089/maps/fb71646388b91922f417a08f7d65f724-120605188d8904c581b71496d5b1ecf2/tiles/{z}/{x}/{y}"
"gee_naip_2004","https://earthengine.googleapis.com/v1/projects/676236615089/maps/4ac3fd24bb4117dda66cc656e0d395af-026b73f791a96d81113e98165d71692c/tiles/{z}/{x}/{y}"
"gee_naip_2005","https://earthengine.googleapis.com/v1/projects/676236615089/maps/d1229bee9e355d8de65308e5e9874b37-341725a1215c3d67e2c908e51f598506/tiles/{z}/{x}/{y}"
"gee_naip_2006","https://earthengine.googleapis.com/v1/projects/676236615089/maps/9d85ff18fd3ea1ff6517171385fade73-e8112587b923d77315703ed57d491f5a/tiles/{z}/{x}/{y}"
"gee_naip_2009","https://earthengine.googleapis.com/v1/projects/676236615089/maps/e9b32635b8b2add3113fcabe187d3950-684b4a975073a9e8a3d85b28df61778b/tiles/{z}/{x}/{y}"
"gee_naip_ir_2009","https://earthengine.googleapis.com/v1/projects/676236615089/maps/b6ae